In [1]:
import os
import glob
import re
import pandas as pd
import numpy as np
import scipy.integrate as integrate

print("🚀 启动【物理严谨终极版】LFP 12维健康特征极速提取引擎 (含时间探针)...\n")

# ==========================================
# 0. 底层物理计算函数 
# ==========================================
def process_cycle(cycle_df):
    c_df = cycle_df.copy()
    if c_df.empty: return c_df, c_df, c_df
    
    charge = c_df[c_df['Status'] == 'Charge'].copy()
    if not charge.empty:
        charge['Time_Sec'] = charge['dt'].cumsum()
        charge['Capacity_Ah'] = integrate.cumulative_trapezoid(np.abs(charge['Current']), charge['Time_Sec'], initial=0) / 3600.0

    discharge = c_df[c_df['Status'] == 'Discharge'].copy()
    if not discharge.empty:
        discharge['Time_Sec'] = discharge['dt'].cumsum()
        discharge['Capacity_Ah'] = integrate.cumulative_trapezoid(np.abs(discharge['Current']), discharge['Time_Sec'], initial=0) / 3600.0
            
    return c_df, charge, discharge

def get_monotonic_data(v, t, q):
    idx = np.argsort(v)
    v_u, u_idx = np.unique(v[idx], return_index=True)
    return v_u, t[idx][u_idx], q[idx][u_idx]

def estimate_dcir(cycle_dis_df):
    if len(cycle_dis_df) > 5:
        v_drop = cycle_dis_df['Voltage'].iloc[0] - cycle_dis_df['Voltage'].iloc[5]
        i_step = abs(cycle_dis_df['Current'].iloc[5])
        return v_drop / (i_step + 1e-6)
    return 0

# ==========================================
# 1. 文件名正则表达式工况解析器
# ==========================================
def extract_protocol_from_filenames(valid_files):
    for f in valid_files:
        fname = os.path.basename(f).upper()
        c_rate_match = re.search(r'(\d+(\.\d+)?C)', fname)
        temp_match = re.search(r'(\d+DEG)', fname)
        soc_match = re.search(r'(\d+(_|-)?\d*SOC)', fname)
        dod_match = re.search(r'(\d+(_|-)?\d*DOD)', fname)
        
        extracted = []
        if c_rate_match: extracted.append(c_rate_match.group(1))
        if temp_match: extracted.append(temp_match.group(1))
        if soc_match: extracted.append(soc_match.group(1))
        if dod_match: extracted.append(dod_match.group(1))
        
        if extracted: return "_".join(extracted)
    return "Protocol_Not_Found"

# ==========================================
# 2. 核心特征提取逻辑 (自带绝对时间探针)
# ==========================================
def extract_features_for_battery(folder_path, battery_name):
    print(f"[{battery_name}] 📂 正在扫描目录...")
    
    valid_files = [f for f in glob.glob(os.path.join(folder_path, "*.parquet")) 
                   if 'standard' not in f.lower() 
                   and '_cu_' not in f.lower() 
                   and 'init' not in f.lower()]
    valid_files.sort()
    
    if not valid_files:
        return None

    experiment_protocol = extract_protocol_from_filenames(valid_files)

    print(f"[{battery_name}] 🔗 正在极速合并 {len(valid_files)} 个切片...")
    df_list = [pd.read_parquet(f) for f in valid_files]
    big_df = pd.concat(df_list, ignore_index=True)

    rename_dict = {'Zeit': 'Date_Time', 'Spannung': 'Voltage', 'Strom': 'Current'}
    found_temp_col = next((col for col in big_df.columns if col.lower() in ['t1', 't2', 'temp', 'temperature'] or 'temp' in col.lower()), None)
    if found_temp_col: rename_dict[found_temp_col] = 'Temperature'
    else: big_df['Temperature'] = 25.0 

    big_df = big_df.rename(columns=rename_dict)
    big_df['Date_Time'] = pd.to_datetime(big_df['Date_Time'], utc=True)
    big_df = big_df.sort_values('Date_Time').reset_index(drop=True)

    big_df['Status'] = 'Rest'
    big_df.loc[big_df['Current'] > 0.005, 'Status'] = 'Charge'
    big_df.loc[big_df['Current'] < -0.005, 'Status'] = 'Discharge'

    is_charge = big_df['Status'] == 'Charge'
    big_df['Global_Cycle'] = (is_charge & (~is_charge.shift(1, fill_value=False))).cumsum()

    big_df['dt'] = big_df['Date_Time'].diff().dt.total_seconds().fillna(0).clip(upper=60)
    big_df['Abs_Current'] = big_df['Current'].abs()

    print(f"[{battery_name}] 🛡️ 启动向量化引擎：正在执行超强清洗...")
    cycle_grp = big_df.groupby(['Global_Cycle', 'Status'])
    metrics = cycle_grp.agg(mean_current=('Abs_Current', 'mean'), duration_sec=('dt', 'sum')).unstack('Status')
    metrics.columns = [f"{col[0]}_{col[1]}" for col in metrics.columns]
    
    try:
        # 🌟 史上最严物理壁垒 🌟
        # 充电电流 > 1.5A 且 时间必须 > 300秒 (5分钟) -> 绝对秒杀 HPPC 脉冲！
        # 放电电流 > 0.8A 且 时间必须 > 300秒 (5分钟) -> 绝对秒杀所有慢速容量标定和短脉冲！
        valid_mask = (
            (metrics.get('mean_current_Charge', 0) > 1.5) & 
            (metrics.get('mean_current_Discharge', 0) > 0.8) & 
            (metrics.get('duration_sec_Charge', 0) > 300) &  
            (metrics.get('duration_sec_Discharge', 0) > 300)
        )
        pure_aging_cycles = metrics[valid_mask].index.tolist()
    except KeyError:
        pure_aging_cycles = []

    if not pure_aging_cycles:
        print(f"[{battery_name}] ❌ 未找到任何纯净循环，放弃提取。")
        return None

    # ==========================================
    # ⏱️ 绝对时间探针：向你证明我们跳过了多少时间！
    # ==========================================
    first_pure_cycle = pure_aging_cycles[0]
    exp_start_time = big_df['Date_Time'].iloc[0]
    true_cycle_start_time = big_df[big_df['Global_Cycle'] == first_pure_cycle]['Date_Time'].iloc[0]
    elapsed_hours_before_aging = (true_cycle_start_time - exp_start_time).total_seconds() / 3600.0
    skipped_garbage_jumps = first_pure_cycle - 1
    true_cycles_count = len(pure_aging_cycles)
    
    print(f"[{battery_name}] 🗑️ 成功跨越：自动剔除了前 {skipped_garbage_jumps} 次充放电跳变！")
    print(f"[{battery_name}] ⏰ 时间铁证：剔除了足足 {elapsed_hours_before_aging:.1f} 小时的预充放与脉冲垃圾数据！")
    print(f"[{battery_name}] 🎯 锁定起点：完美的 3C 老化是从绝对时间第 {elapsed_hours_before_aging:.1f} 小时正式开始的！")

    if true_cycles_count < 100:
        print(f"[{battery_name}] ❌ 纯净循环不足 100 次，放弃提取。")
        return None

    early_cycles = pure_aging_cycles[:105]
    early_df = big_df[big_df['Global_Cycle'].isin(early_cycles)].copy()

    idx_2, idx_10, idx_100 = pure_aging_cycles[1], pure_aging_cycles[9], pure_aging_cycles[99]
    _, _, c2_dis = process_cycle(early_df[early_df['Global_Cycle'] == idx_2])
    _, c10_chg, c10_dis = process_cycle(early_df[early_df['Global_Cycle'] == idx_10])
    _, c100_chg, c100_dis = process_cycle(early_df[early_df['Global_Cycle'] == idx_100])

    features = {}
    features['0_Experiment_Protocol'] = experiment_protocol
    features['F1_Charge_Time_Diff'] = (c100_chg['Time_Sec'].max() if not c100_chg.empty else 0) - (c10_chg['Time_Sec'].max() if not c10_chg.empty else 0)
    features['F8_Discharge_Time_Diff'] = (c100_dis['Time_Sec'].max() if not c100_dis.empty else 0) - (c10_dis['Time_Sec'].max() if not c10_dis.empty else 0)
    
    v_min = max(c10_dis['Voltage'].min(), c100_dis['Voltage'].min())
    v_max = min(c10_dis['Voltage'].max(), c100_dis['Voltage'].max())
    common_V = np.linspace(v_min, v_max, 100)
    
    v10_u, t10_u, q10_u = get_monotonic_data(c10_dis['Voltage'].values, c10_dis['Time_Sec'].values, c10_dis['Capacity_Ah'].values)
    v100_u, t100_u, q100_u = get_monotonic_data(c100_dis['Voltage'].values, c100_dis['Time_Sec'].values, c100_dis['Capacity_Ah'].values)
    
    T_10_interp, T_100_interp = np.interp(common_V, v10_u, t10_u), np.interp(common_V, v100_u, t100_u)
    Q_10_interp, Q_100_interp = np.interp(common_V, v10_u, q10_u), np.interp(common_V, v100_u, q100_u)
    
    features['F7_IC_Area_Diff'] = integrate.trapezoid(np.abs(np.gradient(Q_100_interp, common_V)), common_V) - integrate.trapezoid(np.abs(np.gradient(Q_10_interp, common_V)), common_V)
    features['F9_Var_Time_Diff'] = np.var(T_100_interp - T_10_interp)
    features['F17_Mean_Cap_Diff'] = np.mean(Q_100_interp - Q_10_interp)
    
    cap_hist = []
    for c_idx in pure_aging_cycles[79:100]:
        c_d = early_df[(early_df['Global_Cycle'] == c_idx) & (early_df['Status'] == 'Discharge')].copy()
        if not c_d.empty:
            t_sec = c_d['dt'].cumsum()
            cap_hist.append(integrate.trapezoid(np.abs(c_d['Current']), t_sec) / 3600.0)
            
    x_cyc = np.arange(1, len(cap_hist) + 1)
    features['F22_Linear_Intercept'] = np.polyfit(x_cyc, cap_hist, 1)[1] if len(cap_hist) > 1 else 0
    features['F24_Sqrt_Intercept'] = np.polyfit(np.sqrt(x_cyc), cap_hist, 1)[1] if len(cap_hist) > 1 else 0
    
    temp_2, temp_100 = c2_dis['Temperature'].values, c100_dis['Temperature'].values
    time_2, time_100 = c2_dis['Time_Sec'].values, c100_dis['Time_Sec'].values
    
    features['F34_Temp_Time_Integral_Diff'] = (integrate.trapezoid(temp_100, time_100) if len(temp_100)>1 else 0) - (integrate.trapezoid(temp_2, time_2) if len(temp_2)>1 else 0)
    features['F35_Min_Temp_Diff'] = (np.min(temp_100) if len(temp_100)>0 else 0) - (np.min(temp_2) if len(temp_2)>0 else 0)
    
    min_t, avg_t = [], []
    for c_idx in pure_aging_cycles[1:100]:
        t_s = early_df[(early_df['Global_Cycle'] == c_idx) & (early_df['Status'] == 'Discharge')]['Temperature']
        if not t_s.empty: min_t.append(t_s.min()); avg_t.append(t_s.mean())
            
    features['F38_Avg_Min_Temp'] = np.mean(min_t) if min_t else 0
    features['F39_Avg_Mean_Temp'] = np.mean(avg_t) if avg_t else 0
    features['F41_DCIR_Diff'] = estimate_dcir(c100_dis) - estimate_dcir(c2_dis)
    
    features['Y_Total_Cycles'] = true_cycles_count
    
    print(f"[{battery_name}] ✅ 12维特征精准提取完毕！\n")
    return features

# ==========================================
# 3. 批量执行与总表输出
# ==========================================
base_dir = "." 
all_battery_features = {}

if os.path.exists(base_dir):
    for item in os.listdir(base_dir):
        folder_path = os.path.join(base_dir, item)
        if os.path.isdir(folder_path) and item.startswith("METABatt"):
            battery_name = item.split("_")[-1] 
            feats = extract_features_for_battery(folder_path, battery_name)
            if feats:
                all_battery_features[battery_name] = feats
else:
    print(f"❌ 找不到目录，请确保脚本直接存放在数据文件夹中。")

if all_battery_features:
    final_df = pd.DataFrame.from_dict(all_battery_features, orient='index')
    final_df.index.name = 'Battery_ID'
    
    print("\n" + "="*90)
    print("🏆 LFP 源域 12维物理严谨特征矩阵提取完毕！(转置预览)")
    print("="*90)
    print(final_df.T.to_string())
    
    save_path = "LFP_12D_Features_Strict_Final.csv"
    final_df.to_csv(save_path)
    print(f"\n💾 数据已成功保存至当前目录: {save_path}")

🚀 启动【物理严谨终极版】LFP 12维健康特征极速提取引擎 (含时间探针)...



In [2]:
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt

print("🕵️‍♂️ 正在启动【纯净原始数据流】全景时间轴验证引擎...\n")

# 使用原生的 Matplotlib 样式
plt.style.use('bmh') 
plt.rcParams['font.sans-serif'] = ['Arial']
plt.rcParams['axes.unicode_minus'] = False

def load_raw_stream_data(base_dir, target, num_files_to_load=5):
    """
    严格排除 standard 和 _cu_ 体检文件，按时间顺序拼接指定数量的原始循环文件，
    不做任何圈数打标，直接拉出绝对时间轴。
    """
    # 1. 定位电池文件夹
    folders = [f for f in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, f)) and target in f]
    if not folders:
        print(f"❌ 未找到包含 {target} 的文件夹")
        return None
    folder_path = os.path.join(base_dir, folders[0])
    
    # 2. 严格过滤文件名：排除体检，保留所有循环与初始化数据
    all_files = glob.glob(os.path.join(folder_path, "*.parquet"))
    valid_files = [f for f in all_files if 'standard' not in f.lower() and '_cu_' not in f.lower()]
    valid_files.sort()  # 🌟 极其关键：按设备生成的时间戳先后顺序进行绝对排序
    
    if not valid_files:
        print(f"❌ 在 {target} 文件夹中未找到有效的循环文件")
        return None
        
    print(f"[{target}] 文件夹内共有 {len(valid_files)} 个非体检 Parquet 文件。")
    print(f"[{target}] 正在为您拼装前 {num_files_to_load} 个文件的原始物理数据流...")
    
    # 3. 循环读取并极速拼接
    df_list = []
    for f in valid_files[:num_files_to_load]:
        print(f"   -> 正在并入数据流: {os.path.basename(f)}")
        df_list.append(pd.read_parquet(f))
        
    df = pd.concat(df_list, ignore_index=True)
    
    # 4. 统一工况列名
    rename_dict = {'Zeit': 'Date_Time', 'Spannung': 'Voltage', 'Strom': 'Current'}
    df = df.rename(columns=rename_dict)
    df['Date_Time'] = pd.to_datetime(df['Date_Time'], utc=True)
    df = df.sort_values('Date_Time').reset_index(drop=True)
    
    # 5. 💡 核心黑科技：抛弃圈数，直接计算从实验第一秒开始的“绝对连续小时数”
    df['Relative_Time_Hours'] = (df['Date_Time'] - df['Date_Time'].iloc[0]).dt.total_seconds() / 3600.0
    
    return df

# 执行数据流拼装（默认读前5个文件，你可以根据电脑配置调大，比如 10 或者 23）
df_007 = load_raw_stream_data(".", "007", num_files_to_load=5)
df_022 = load_raw_stream_data(".", "022", num_files_to_load=5)

if df_007 is not None and df_022 is not None:
    # 建立双子图大画布
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 9), sharex=True)
    
    # 📈 --- 上图：连续电流时间轴 ---
    ax1.plot(df_007['Relative_Time_Hours'], df_007['Current'], label='Battery 007 (40%-100% Window)', color='#d62728', linewidth=0.8, alpha=0.9)
    ax1.plot(df_022['Relative_Time_Hours'], df_022['Current'], label='Battery 022 (0%-60% Window)', color='#1f77b4', linewidth=0.8, linestyle='--', alpha=0.7)
    ax1.set_ylabel('Current (A)', fontsize=14, fontweight='bold')
    ax1.set_title('Continuous Telemetry: Current Timeline (No Cycle Assumptions)', fontsize=16, fontweight='bold')
    ax1.legend(fontsize=12, loc='upper right')
    
    # 📉 --- 下图：连续电压时间轴 ---
    ax2.plot(df_007['Relative_Time_Hours'], df_007['Voltage'], color='#d62728', linewidth=0.8, alpha=0.9)
    ax2.plot(df_022['Relative_Time_Hours'], df_022['Voltage'], color='#1f77b4', linewidth=0.8, linestyle='--', alpha=0.7)
    ax2.set_xlabel('Continuous Experiment Elapsed Time (Hours)', fontsize=14, fontweight='bold')
    ax2.set_ylabel('Voltage (V)', fontsize=14, fontweight='bold')
    ax2.set_title('Continuous Telemetry: Voltage Timeline (No Cycle Assumptions)', fontsize=16, fontweight='bold')
    
    # 加上电池物理强力边界参考线
    ax2.axhline(y=3.6, color='black', linestyle=':', linewidth=1.5)
    ax2.axhline(y=2.0, color='black', linestyle=':', linewidth=1.5)
    ax2.text(0.5, 3.63, '100% SOC Bound (3.6V)', color='black', fontsize=11, fontweight='bold')
    ax2.text(0.5, 1.85, '0% SOC Bound (2.0V)', color='black', fontsize=11, fontweight='bold')

    plt.tight_layout()
    output_filename = "007_vs_022_Continuous_Raw_Stream.png"
    plt.savefig(output_filename, dpi=300)
    print(f"\n📊 原始流拼接绘图成功！全景高清图片已保存至: {output_filename}")
    plt.show()
else:
    print("❌ 数据流组装失败，请检查目录。")

🕵️‍♂️ 正在启动【纯净原始数据流】全景时间轴验证引擎...

❌ 未找到包含 007 的文件夹
❌ 未找到包含 022 的文件夹
❌ 数据流组装失败，请检查目录。


In [ ]:
import pandas as pd
import numpy as np
from sklearn.svm import SVR
import matplotlib.pyplot as plt
import seaborn as sns

print("⚔️ 启动 [1v1 极限预测引擎]: 用 007 训练，预测 022...\n")

sns.set_theme(style="whitegrid")
plt.rcParams['font.sans-serif'] = ['Arial']
plt.rcParams['axes.unicode_minus'] = False

# 1. 加载特征表
try:
    df = pd.read_csv("LFP_12D_Features_Strict_Final.csv", index_col=0)
    df.index = df.index.astype(str)
except FileNotFoundError:
    print("❌ 找不到特征文件！")
    exit()

if '7' not in df.index or '22' not in df.index:
    print("❌ 数据表中缺少 007 (索引'7') 或 022 (索引'22') 的数据！")
    import sys; sys.exit()

# 2. 严格隔离：007 为唯一训练集，022 为唯一测试集
train_id, test_id = '7', '22'
train_df = df.loc[[train_id]].copy()
test_df = df.loc[[test_id]].copy()

feature_cols = [col for col in df.columns if col.startswith('F')]

X_train = train_df[feature_cols].values
y_train = train_df['Y_Total_Cycles'].values
X_test = test_df[feature_cols].values
y_test = test_df['Y_Total_Cycles'].values

print(f"📖 教材库：只有 1 块电池 (007, 真实寿命 {y_train[0]} 圈)")
print(f"📝 考试卷：只有 1 块电池 (022, 完全未知工况)")

# 3. 放弃 Z-score (因为 1 个样本的方差为 0，数学上无法进行标准化)
# 采用极其原始的绝对值直接输入 SVM
svm_model = SVR(kernel='rbf', C=1000, gamma='scale')
svm_model.fit(X_train, y_train)

# 4. 执行预测
y_pred = svm_model.predict(X_test)

print("\n" + "="*50)
print("🏆 1v1 极限预测结果揭晓！")
print("="*50)
actual = y_test[0]
predicted = y_pred[0]
error = abs(actual - predicted)
error_pct = (error / actual) * 100

print(f"🔋 目标测试电池: 022 (工况: {test_df.loc[test_id, '0_Experiment_Protocol']})")
print(f"   -> 真实寿命: {actual:.0f} 圈")
print(f"   -> AI 预测:  {predicted:.0f} 圈")
print(f"   -> 绝对误差: {error:.0f} 圈 ({error_pct:.2f}%)")
print("-" * 50)

# 5. 画图
plot_data = [
    {'Battery': '022', 'Value': actual, 'Type': 'Actual Lifetime'},
    {'Battery': '022', 'Value': predicted, 'Type': 'SVM Predicted (Trained only on 007)'}
]

results_plot_df = pd.DataFrame(plot_data)

plt.figure(figsize=(7, 6))
ax = sns.barplot(x='Battery', y='Value', hue='Type', data=results_plot_df, palette=['#34495e', '#e74c3c'])

for p in ax.patches:
    ax.annotate(format(p.get_height(), '.0f'), 
                (p.get_x() + p.get_width() / 2., p.get_height()), 
                ha = 'center', va = 'center', 
                xytext = (0, 9), 
                textcoords = 'offset points',
                fontsize=12, fontweight='bold')

plt.title('1v1 Blind Test: Predict 022 using ONLY 007', fontsize=15, fontweight='bold', pad=20)
plt.ylabel('Lifetime (Cycles)', fontsize=14, fontweight='bold')
plt.ylim(0, max(actual, predicted) * 1.2) 
plt.legend(title='Category', loc='upper center')

plt.tight_layout()
plt.savefig("1v1_Prediction_007_to_022.png", dpi=300)
print("\n📊 图表已保存！请查看 AI 在极度偏科时的瞎猜结果。")
plt.show()

⚔️ 启动 [1v1 极限预测引擎]: 用 007 训练，预测 022...

❌ 找不到特征文件！


NameError: name 'df' is not defined

: 

In [ ]:
import pandas as pd

# 把你的账号密码填进来
storage_options = {
    "key": "你的登录账号", 
    "secret": "你的登录密码",
    # 注意：API 的端口通常是 9000，网页端是 9001。你可以先试 9000。
    "client_kwargs": {"endpoint_url": "https://iseadocker.isea.rwth-aachen.de:9000"}
}

# 尝试隔空读取网络上的文件
# S3 的路径通常去掉了网址部分，从 bucket 名字（如 projects）开始写
df = pd.read_parquet(
    "s3://projects/j8005-metabatt/Metabatt/A123/METABatt_A123_APR18650M1B_003/你要读取的文件名.parquet",
    storage_options=storage_options
)
print(df.head())

In [ ]:
%pip install scipy scikit-learn matplotlib seaborn

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Program Files\Python 3.10.2\python.exe -m pip install --upgrade pip' command.


In [4]:
import s3fs

# 终极网络通讯配置字典 (请填入你在网页端生成的真实密钥！)
storage_options = {
    "key": "wKuLBjgZPfsOJrJfQPNK",     
    "secret": "JMSsRN5fGcUdcbgSrixhBrpJZKbS6WZqKzdlJipp",  
    "client_kwargs": {
        "endpoint_url": "https://iseadocker.isea.rwth-aachen.de:9000",
        "region_name": "us-east-1" 
    },
    "config_kwargs": {
        "s3": {"addressing_style": "path"}, 
        "signature_version": "s3v4"         
    }
}

# 实例化 S3 文件系统探针
fs = s3fs.S3FileSystem(**storage_options)

# 把你的网页链接翻译成了真实的 S3 目标路径
target_folder = "projects/j8005-metabatt/Metabatt/A123/"

print(f"正在跨网扫描 RWTH 服务器上的目录: {target_folder} ...\n")

try:
    # 尝试列出该文件夹下的所有内容
    folder_contents = fs.ls(target_folder)
    print("✅ 鉴权通过！成功连上底层数据库！A123 目录下的内容有：")
    
    # 打印前 10 个子文件夹/文件看看
    for item in folder_contents[:10]:
        print(f"  - {item}")
        
    print(f"\n(共扫描到 {len(folder_contents)} 个项目)")
    
except Exception as e:
    print("❌ 连接被拒绝，请检查报错细节：")
    print(e)

正在跨网扫描 RWTH 服务器上的目录: projects/j8005-metabatt/Metabatt/A123/ ...

✅ 鉴权通过！成功连上底层数据库！A123 目录下的内容有：
  - projects/j8005-metabatt/Metabatt/A123/10_TRACY
  - projects/j8005-metabatt/Metabatt/A123/20_export_pulse
  - projects/j8005-metabatt/Metabatt/A123/30_export_qocv
  - projects/j8005-metabatt/Metabatt/A123/40_capacity_monitore
  - projects/j8005-metabatt/Metabatt/A123/BRONZE_CU
  - projects/j8005-metabatt/Metabatt/A123/METABatt_A123_APR18650M1B_003
  - projects/j8005-metabatt/Metabatt/A123/METABatt_A123_APR18650M1B_004
  - projects/j8005-metabatt/Metabatt/A123/METABatt_A123_APR18650M1B_006
  - projects/j8005-metabatt/Metabatt/A123/METABatt_A123_APR18650M1B_007
  - projects/j8005-metabatt/Metabatt/A123/METABatt_A123_APR18650M1B_008

(共扫描到 428 个项目)


In [9]:
import s3fs
import os
import re
import pandas as pd
import numpy as np
import scipy.integrate as integrate
import warnings

warnings.filterwarnings('ignore')

print("🚀 启动【物理严谨终极版】LFP 12维健康特征极速提取引擎 (S3云端直连版)...\n")

# ==========================================
# 0. S3 云端底层鉴权配置 (请换上你的真实密钥)
# ==========================================
storage_options = {
    "key": "wKuLBjgZPfsOJrJfQPNK",     
    "secret": "JMSsRN5fGcUdcbgSrixhBrpJZKbS6WZqKzdlJipp",  
    "client_kwargs": {
        "endpoint_url": "https://iseadocker.isea.rwth-aachen.de:9000",
        "region_name": "us-east-1" 
    },
    "config_kwargs": {
        "s3": {"addressing_style": "path"}, 
        "signature_version": "s3v4"         
    }
}
fs = s3fs.S3FileSystem(**storage_options)
target_folder = "projects/j8005-metabatt/Metabatt/A123/"

# ==========================================
# 1. 底层物理计算函数 (完全保留你的原版逻辑)
# ==========================================
def process_cycle(cycle_df):
    c_df = cycle_df.copy()
    if c_df.empty: return c_df, c_df, c_df
    
    charge = c_df[c_df['Status'] == 'Charge'].copy()
    if not charge.empty:
        charge['Time_Sec'] = charge['dt'].cumsum()
        charge['Capacity_Ah'] = integrate.cumulative_trapezoid(np.abs(charge['Current']), charge['Time_Sec'], initial=0) / 3600.0

    discharge = c_df[c_df['Status'] == 'Discharge'].copy()
    if not discharge.empty:
        discharge['Time_Sec'] = discharge['dt'].cumsum()
        discharge['Capacity_Ah'] = integrate.cumulative_trapezoid(np.abs(discharge['Current']), discharge['Time_Sec'], initial=0) / 3600.0
            
    return c_df, charge, discharge

def get_monotonic_data(v, t, q):
    idx = np.argsort(v)
    v_u, u_idx = np.unique(v[idx], return_index=True)
    return v_u, t[idx][u_idx], q[idx][u_idx]

def estimate_dcir(cycle_dis_df):
    if len(cycle_dis_df) > 5:
        v_drop = cycle_dis_df['Voltage'].iloc[0] - cycle_dis_df['Voltage'].iloc[5]
        i_step = abs(cycle_dis_df['Current'].iloc[5])
        return v_drop / (i_step + 1e-6)
    return 0

def extract_protocol_from_filenames(valid_files):
    for f in valid_files:
        fname = os.path.basename(f).upper()
        c_rate_match = re.search(r'(\d+(\.\d+)?C)', fname)
        temp_match = re.search(r'(\d+DEG)', fname)
        soc_match = re.search(r'(\d+(_|-)?\d*SOC)', fname)
        dod_match = re.search(r'(\d+(_|-)?\d*DOD)', fname)
        
        extracted = []
        if c_rate_match: extracted.append(c_rate_match.group(1))
        if temp_match: extracted.append(temp_match.group(1))
        if soc_match: extracted.append(soc_match.group(1))
        if dod_match: extracted.append(dod_match.group(1))
        
        if extracted: return "_".join(extracted)
    return "Protocol_Not_Found"

# ==========================================
# 2. 核心特征提取逻辑 (云端适配版)
# ==========================================
def extract_features_for_battery(s3_files, battery_name):
    print(f"[{battery_name}] 🔗 正在从 S3 极速拉取并合并 {len(s3_files)} 个切片...")
    
    # 【改动点】通过 s3 路径隔空读取 parquet
    df_list = [pd.read_parquet(f"s3://{f}", storage_options=storage_options) for f in s3_files]
    big_df = pd.concat(df_list, ignore_index=True)

    rename_dict = {'Zeit': 'Date_Time', 'Spannung': 'Voltage', 'Strom': 'Current'}
    found_temp_col = next((col for col in big_df.columns if col.lower() in ['t1', 't2', 'temp', 'temperature'] or 'temp' in col.lower()), None)
    if found_temp_col: rename_dict[found_temp_col] = 'Temperature'
    else: big_df['Temperature'] = 25.0 

    big_df = big_df.rename(columns=rename_dict)
    big_df['Date_Time'] = pd.to_datetime(big_df['Date_Time'], utc=True)
    big_df = big_df.sort_values('Date_Time').reset_index(drop=True)

    big_df['Status'] = 'Rest'
    big_df.loc[big_df['Current'] > 0.005, 'Status'] = 'Charge'
    big_df.loc[big_df['Current'] < -0.005, 'Status'] = 'Discharge'

    is_charge = big_df['Status'] == 'Charge'
    big_df['Global_Cycle'] = (is_charge & (~is_charge.shift(1, fill_value=False))).cumsum()

    big_df['dt'] = big_df['Date_Time'].diff().dt.total_seconds().fillna(0).clip(upper=60)
    big_df['Abs_Current'] = big_df['Current'].abs()

    print(f"[{battery_name}] 🛡️ 启动向量化引擎：正在执行超强物理清洗...")
    cycle_grp = big_df.groupby(['Global_Cycle', 'Status'])
    metrics = cycle_grp.agg(mean_current=('Abs_Current', 'mean'), duration_sec=('dt', 'sum')).unstack('Status')
    metrics.columns = [f"{col[0]}_{col[1]}" for col in metrics.columns]
    
    try:
        valid_mask = (
            (metrics.get('mean_current_Charge', 0) > 1.5) & 
            (metrics.get('mean_current_Discharge', 0) > 0.8) & 
            (metrics.get('duration_sec_Charge', 0) > 300) &  
            (metrics.get('duration_sec_Discharge', 0) > 300)
        )
        pure_aging_cycles = metrics[valid_mask].index.tolist()
    except KeyError:
        pure_aging_cycles = []

    if not pure_aging_cycles:
        print(f"[{battery_name}] ❌ 未找到任何纯净循环，放弃提取。")
        return None

    first_pure_cycle = pure_aging_cycles[0]
    exp_start_time = big_df['Date_Time'].iloc[0]
    true_cycle_start_time = big_df[big_df['Global_Cycle'] == first_pure_cycle]['Date_Time'].iloc[0]
    elapsed_hours_before_aging = (true_cycle_start_time - exp_start_time).total_seconds() / 3600.0
    skipped_garbage_jumps = first_pure_cycle - 1
    true_cycles_count = len(pure_aging_cycles)
    
    print(f"[{battery_name}] 🗑️ 成功跨越：自动剔除了前 {skipped_garbage_jumps} 次充放电跳变！")
    print(f"[{battery_name}] ⏰ 时间铁证：剔除了足足 {elapsed_hours_before_aging:.1f} 小时的预充放与脉冲垃圾数据！")

    if true_cycles_count < 100:
        print(f"[{battery_name}] ❌ 纯净循环不足 100 次，放弃提取。")
        return None

    early_cycles = pure_aging_cycles[:105]
    early_df = big_df[big_df['Global_Cycle'].isin(early_cycles)].copy()

    idx_2, idx_10, idx_100 = pure_aging_cycles[1], pure_aging_cycles[9], pure_aging_cycles[99]
    _, _, c2_dis = process_cycle(early_df[early_df['Global_Cycle'] == idx_2])
    _, c10_chg, c10_dis = process_cycle(early_df[early_df['Global_Cycle'] == idx_10])
    _, c100_chg, c100_dis = process_cycle(early_df[early_df['Global_Cycle'] == idx_100])

    features = {}
    features['0_Experiment_Protocol'] = extract_protocol_from_filenames(s3_files)
    features['F1_Charge_Time_Diff'] = (c100_chg['Time_Sec'].max() if not c100_chg.empty else 0) - (c10_chg['Time_Sec'].max() if not c10_chg.empty else 0)
    features['F8_Discharge_Time_Diff'] = (c100_dis['Time_Sec'].max() if not c100_dis.empty else 0) - (c10_dis['Time_Sec'].max() if not c10_dis.empty else 0)
    
    v_min = max(c10_dis['Voltage'].min(), c100_dis['Voltage'].min())
    v_max = min(c10_dis['Voltage'].max(), c100_dis['Voltage'].max())
    common_V = np.linspace(v_min, v_max, 100)
    
    v10_u, t10_u, q10_u = get_monotonic_data(c10_dis['Voltage'].values, c10_dis['Time_Sec'].values, c10_dis['Capacity_Ah'].values)
    v100_u, t100_u, q100_u = get_monotonic_data(c100_dis['Voltage'].values, c100_dis['Time_Sec'].values, c100_dis['Capacity_Ah'].values)
    
    T_10_interp, T_100_interp = np.interp(common_V, v10_u, t10_u), np.interp(common_V, v100_u, t100_u)
    Q_10_interp, Q_100_interp = np.interp(common_V, v10_u, q10_u), np.interp(common_V, v100_u, q100_u)
    
    features['F7_IC_Area_Diff'] = integrate.trapezoid(np.abs(np.gradient(Q_100_interp, common_V)), common_V) - integrate.trapezoid(np.abs(np.gradient(Q_10_interp, common_V)), common_V)
    features['F9_Var_Time_Diff'] = np.var(T_100_interp - T_10_interp)
    features['F17_Mean_Cap_Diff'] = np.mean(Q_100_interp - Q_10_interp)
    
    cap_hist = []
    for c_idx in pure_aging_cycles[79:100]:
        c_d = early_df[(early_df['Global_Cycle'] == c_idx) & (early_df['Status'] == 'Discharge')].copy()
        if not c_d.empty:
            t_sec = c_d['dt'].cumsum()
            cap_hist.append(integrate.trapezoid(np.abs(c_d['Current']), t_sec) / 3600.0)
            
    x_cyc = np.arange(1, len(cap_hist) + 1)
    features['F22_Linear_Intercept'] = np.polyfit(x_cyc, cap_hist, 1)[1] if len(cap_hist) > 1 else 0
    features['F24_Sqrt_Intercept'] = np.polyfit(np.sqrt(x_cyc), cap_hist, 1)[1] if len(cap_hist) > 1 else 0
    
    temp_2, temp_100 = c2_dis['Temperature'].values, c100_dis['Temperature'].values
    time_2, time_100 = c2_dis['Time_Sec'].values, c100_dis['Time_Sec'].values
    
    features['F34_Temp_Time_Integral_Diff'] = (integrate.trapezoid(temp_100, time_100) if len(temp_100)>1 else 0) - (integrate.trapezoid(temp_2, time_2) if len(temp_2)>1 else 0)
    features['F35_Min_Temp_Diff'] = (np.min(temp_100) if len(temp_100)>0 else 0) - (np.min(temp_2) if len(temp_2)>0 else 0)
    
    min_t, avg_t = [], []
    for c_idx in pure_aging_cycles[1:100]:
        t_s = early_df[(early_df['Global_Cycle'] == c_idx) & (early_df['Status'] == 'Discharge')]['Temperature']
        if not t_s.empty: min_t.append(t_s.min()); avg_t.append(t_s.mean())
            
    features['F38_Avg_Min_Temp'] = np.mean(min_t) if min_t else 0
    features['F39_Avg_Mean_Temp'] = np.mean(avg_t) if avg_t else 0
    features['F41_DCIR_Diff'] = estimate_dcir(c100_dis) - estimate_dcir(c2_dis)
    
    features['Y_Total_Cycles'] = true_cycles_count
    
    print(f"[{battery_name}] ✅ 12维特征精准提取完毕！\n")
    return features

# ==========================================
# 3. 云端目录扫描与批量执行 (强控 80 块电池上限)
# ==========================================
all_items = fs.ls(target_folder)
battery_folders = [item for item in all_items if "METABatt_A123_APR18650M1B_" in item and item[-3:].isdigit()]

# 【改动点】硬性截断：最多只处理前 80 块电池
battery_folders = battery_folders[:80]
print(f"📦 共锁定 {len(battery_folders)} 块目标电池目录，即将开始全速抽取...\n")

all_battery_features = {}

for folder in battery_folders:
    battery_name = folder.split("_")[-1]
    all_files = fs.ls(folder)
    
    # 【改动点】极度严苛的文件过滤器：只要 _Cyc_
    valid_files = [f for f in all_files 
                   if f.endswith('.parquet') 
                   and '_Cyc_' in f 
                   and not any(x in f for x in ['STD', 'EIS', 'standard', '_cu_', 'init'])]
    
    valid_files.sort()
    
    if not valid_files:
        print(f"[{battery_name}] ⚠️ 跳过：未找到有效的 _Cyc_ 循环文件。")
        continue
        
    try:
        feats = extract_features_for_battery(valid_files, battery_name)
        if feats:
            all_battery_features[battery_name] = feats
    except Exception as e:
        print(f"[{battery_name}] ❌ 提取过程中发生异常: {e}")

if all_battery_features:
    final_df = pd.DataFrame.from_dict(all_battery_features, orient='index')
    final_df.index.name = 'Battery_ID'
    
    print("\n" + "="*90)
    print("🏆 LFP 源域 12维物理严谨特征矩阵 (云端 80 块全量版) 提取完毕！(转置预览)")
    print("="*90)
    print(final_df.T.to_string())
    
    save_path = "LFP_12D_Features_Strict_Final.csv"
    final_df.to_csv(save_path)
    print(f"\n💾 数据矩阵已成功落地本地目录: {save_path}")
    print("⚡ 矩阵已就绪，LFP 向 NCA 的迁移学习管道大门正式开启！")
else:
    print("\n⚠️ 扫描结束，但未能成功提取到任何特征矩阵。请检查云端文件结构或物理过滤参数。")

🚀 启动【物理严谨终极版】LFP 12维健康特征极速提取引擎 (S3云端直连版)...

📦 共锁定 80 块目标电池目录，即将开始全速抽取...

[003] ⚠️ 跳过：未找到有效的 _Cyc_ 循环文件。
[004] ⚠️ 跳过：未找到有效的 _Cyc_ 循环文件。
[006] ⚠️ 跳过：未找到有效的 _Cyc_ 循环文件。
[007] 🔗 正在从 S3 极速拉取并合并 22 个切片...
[007] 🛡️ 启动向量化引擎：正在执行超强物理清洗...
[007] 🗑️ 成功跨越：自动剔除了前 2 次充放电跳变！
[007] ⏰ 时间铁证：剔除了足足 22.0 小时的预充放与脉冲垃圾数据！
[007] ✅ 12维特征精准提取完毕！

[008] 🔗 正在从 S3 极速拉取并合并 21 个切片...
[008] 🛡️ 启动向量化引擎：正在执行超强物理清洗...
[008] 🗑️ 成功跨越：自动剔除了前 2 次充放电跳变！
[008] ⏰ 时间铁证：剔除了足足 22.3 小时的预充放与脉冲垃圾数据！
[008] ✅ 12维特征精准提取完毕！

[009] 🔗 正在从 S3 极速拉取并合并 22 个切片...
[009] 🛡️ 启动向量化引擎：正在执行超强物理清洗...
[009] 🗑️ 成功跨越：自动剔除了前 2 次充放电跳变！
[009] ⏰ 时间铁证：剔除了足足 12.6 小时的预充放与脉冲垃圾数据！
[009] ✅ 12维特征精准提取完毕！

[010] 🔗 正在从 S3 极速拉取并合并 22 个切片...
[010] 🛡️ 启动向量化引擎：正在执行超强物理清洗...
[010] 🗑️ 成功跨越：自动剔除了前 2 次充放电跳变！
[010] ⏰ 时间铁证：剔除了足足 12.6 小时的预充放与脉冲垃圾数据！
[010] ✅ 12维特征精准提取完毕！

[011] 🔗 正在从 S3 极速拉取并合并 21 个切片...
[011] 🛡️ 启动向量化引擎：正在执行超强物理清洗...
[011] 🗑️ 成功跨越：自动剔除了前 1 次充放电跳变！
[011] ⏰ 时间铁证：剔除了足足 22.6 小时的预充放与脉冲垃圾数据！
[011] ✅ 12维特征精准提取完毕！

[012] 🔗 正在从 S3 极速拉取并合并 22 个切片...
[012] 🛡️ 启动向量化引擎：正在

In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, r2_score
import warnings

warnings.filterwarnings('ignore')

print("🚀 启动 [全透视版] LFP 源域寿命盲测引擎...\n")

# ==========================================
# 1. 加载数据并准备考卷
# ==========================================
df = pd.read_csv("LFP_12D_Features_Strict_Final.csv", index_col=0)
df_encoded = pd.get_dummies(df, columns=['0_Experiment_Protocol'], drop_first=False)

X = df_encoded.drop(columns=['Y_Total_Cycles'])
y = df_encoded['Y_Total_Cycles']

# 核心隔离区：随机切分 20% 作为绝对没见过的测试集
# random_state=42 保证每次抽出来的都是同一批电池，方便你对照
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"📚 训练集 (供模型寻找规律): {len(X_train)} 块电池")
print(f"🔒 测试集 (模型绝对没见过的考卷): {len(X_test)} 块电池")
print(f"   -> 考卷电池 ID 列表: {y_test.index.tolist()}\n")

# ==========================================
# 2. 尺度对齐与闭卷考试
# ==========================================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test) # 注意：考卷只能用训练集的尺度去缩放，绝对不能透题！

# 训练 Random Forest (这部分模型只看得到训练集的 26 块电池)
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train_scaled, y_train)

# 考卷下发：让模型去盲猜那 7 块陌生电池的寿命
y_pred_rf = rf_model.predict(X_test_scaled)

# ==========================================
# 3. 撕开伪装：真实 vs 预测 一对一公开处刑
# ==========================================
# 把结果打包成一个漂亮的 DataFrame 方便对比
results_df = pd.DataFrame({
    '真实寿命 (True Cycles)': y_test.values,
    '模型盲猜 (Predicted)': np.round(y_pred_rf, 0).astype(int),
}, index=y_test.index)

# 计算绝对误差
results_df['误差 (Error)'] = results_df['模型盲猜 (Predicted)'] - results_df['真实寿命 (True Cycles)']
results_df['误差率 (%)'] = np.round((results_df['误差 (Error)'].abs() / results_df['真实寿命 (True Cycles)']) * 100, 2)

print("⚖️ 【Random Forest 盲测答卷对峙表】")
print("=" * 60)
print(results_df.to_string())
print("=" * 60)

# ==========================================
# 4. 整体打分
# ==========================================
r2 = r2_score(y_test, y_pred_rf)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_rf))

print(f"\n🏆 最终成绩单:")
print(f"   - R² (决定系数): {r2:.3f}  (越接近 1 越牛逼)")
print(f"   - RMSE (均方根误差): {rmse:.1f} 圈 (平均每块电池猜错几圈)")

# 画一张优美的对峙散点图
plt.figure(figsize=(7, 5))
plt.scatter(y_test, y_pred_rf, color='dodgerblue', edgecolor='k', s=80, zorder=3)
# 画出 y=x 的完美预测线
min_val = min(y_test.min(), y_pred_rf.min()) - 500
max_val = max(y_test.max(), y_pred_rf.max()) + 500
plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, zorder=2, label='完美预测线 (y=x)')

# 给每个点标上电池 ID，让你一眼看清是谁拖了后腿
for idx, true_val, pred_val in zip(y_test.index, y_test.values, y_pred_rf):
    plt.text(true_val+50, pred_val-100, f"ID:{idx}", fontsize=9, zorder=4)

plt.xlabel("真实寿命 (True Total Cycles)")
plt.ylabel("模型盲猜 (Predicted Total Cycles)")
plt.title("LFP 陌生电池盲测一对一对比 (Random Forest)")
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6

SyntaxError: '(' was never closed (576318819.py, line 90)